In [1]:
from datasets import load_from_disk

train_dataset = load_from_disk("../data/train_pairs")

print(train_dataset)
print(train_dataset[0])

Dataset({
    features: ['anchor', 'positive', 'intent'],
    num_rows: 10003
})
{'anchor': 'I am still waiting on my card?', 'positive': 'What is the expected delivery date for my card?', 'intent': 'card_arrival'}


In [2]:
train_dataset = train_dataset.select_columns(["anchor", "positive"])

In [3]:
from sentence_transformers import SentenceTransformer
from peft import LoraConfig, TaskType

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value", "key", "dense"],
)

model.add_adapter(peft_config)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable} / {total} ({100 * trainable / total:.4f}%)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable: 675840 / 23389056 (2.8896%)


In [4]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="../training/checkpoints",
    num_train_epochs=15,
    per_device_train_batch_size=64,
    learning_rate=2e-4,
    eval_strategy="no",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    load_best_model_at_end=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [5]:
import os
from huggingface_hub import login

hf_token = os.getenv("HUGGING_FACE_ACCESS_TOKEN")

login(token=hf_token)

In [6]:
trainer.train()

Step,Training Loss
10,2.368894
20,2.100580
30,2.047520
40,1.902055
50,1.829619
60,1.806055
70,1.755806
80,1.720811
90,1.618919
100,1.586040


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2355, training_loss=0.9849543681823018, metrics={'train_runtime': 146.6539, 'train_samples_per_second': 1023.123, 'train_steps_per_second': 16.058, 'total_flos': 0.0, 'train_loss': 0.9849543681823018, 'epoch': 15.0})

In [7]:
print(type(model[0].auto_model))
model.save_pretrained("../training/model")
print("Saved!")

<class 'transformers.models.bert.modeling_bert.BertModel'>


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!


In [8]:
from sentence_transformers import SentenceTransformer
from peft import PeftModel
import numpy as np

base = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
finetuned = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

base_transformer = finetuned[0].auto_model
peft_model = PeftModel.from_pretrained(base_transformer, "../training/model")
print(type(peft_model))

merged = peft_model.merge_and_unload()
print(type(merged))

finetuned[0].auto_model = merged

test = ["I lost my card and need a replacement"]
base_emb = base.encode(test, normalize_embeddings=True)
ft_emb = finetuned.encode(test, normalize_embeddings=True)

print("Same?", np.allclose(base_emb, ft_emb, atol=1e-5))
print("Max diff:", np.abs(base_emb - ft_emb).max())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'peft.peft_model.PeftModelForFeatureExtraction'>
<class 'transformers.models.bert.modeling_bert.BertModel'>
Same? False
Max diff: 0.165728


In [9]:
import os
for f in os.walk("../training/model"):
    print(f)

('../training/model', ['1_Pooling', '2_Normalize'], ['adapter_config.json', 'adapter_model.safetensors', 'config_sentence_transformers.json', 'modules.json', 'README.md', 'sentence_bert_config.json', 'tokenizer.json', 'tokenizer_config.json'])
('../training/model\\1_Pooling', [], ['config.json'])
('../training/model\\2_Normalize', [], [])


In [10]:
print(type(finetuned[0].auto_model))

<class 'transformers.models.bert.modeling_bert.BertModel'>
